# Build a SCHISM workspace procedurally

**Learning goals:** Construct a SCHISM grid and configuration in Python and generate a workspace without launching SCHISM.

**Prerequisites:** Lesson 1; install `rompy-schism`. A SCHISM binary is not required for workspace generation.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_01_rompy_orientation](journey_01_rompy_orientation/)

Next: [journey_03_schism_grid_data](journey_03_schism_grid_data/)


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid, SCHISMConfig
from rompy_schism.data import SCHISMData
from rompy_schism.namelists import NML
from rompy_schism.namelists.param import Param, Core
from rompy.model import ModelRun

# A real mesh and vertical grid are enough to demonstrate offline workspace generation.
root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "tests" / "data" / "schism").is_dir())
case = root / "tests" / "data" / "schism"
grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
config = SCHISMConfig(
    grid=grid,
    data=SCHISMData(),
    nml=NML(param=Param(core=Core(rnday=0.01))),
)
with TemporaryDirectory() as output:
    run = ModelRun(
        run_id="procedural",
        output_dir=output,
        period={"start": "2023-01-01", "end": "2023-01-01T01:00", "interval": "1h"},
        config=config,
    )
    workspace = Path(run.generate())
    print("Generated workspace:", workspace)
    print("Generated files:", len(list(workspace.iterdir())))
